# 00 — Setup

Creates the four objects the reference runs on. Nothing is loaded here; the rows come from the next notebook.

| Object | What it is |
| --- | --- |
| `bt_batch_id` | One-row counter. Incremented inside the load transaction, so an empty load still gets an id. |
| `bt_orders` | Synthetic order lines. Business key is `(order_id, line_no)`. `load_ts` is not part of the key. |
| `bt_load_ledger` | One row per attempt, including a load of zero rows. |
| `bt_orders_latest_batch` | Latest attempt only, read from `MAX(batch_id)` on the ledger. |
| `bt_orders_latest_freshness` | The same latest attempt, kept as its own object so its schedule is independent. |

`MAX(batch_id)` comes from the ledger on purpose. Reading it off `bt_orders` would skip an empty load that deleted every order row.

## Session

Notebooks do not share session variables the way `snow sql -f` does, so the database and schema are written out here instead of read from `$demo_db`. These are the only two names to change if you point the notebooks somewhere else.

The role needs `EXECUTE DATA METRIC FUNCTION` on the account to call the system DMFs later. This repo does not grant it.

In [ ]:
USE ROLE BT_BUILDER_ROLE;
USE WAREHOUSE BT_WH;
USE DATABASE BT_DQ_REF;
CREATE SCHEMA IF NOT EXISTS BT_DQ_REF;
USE SCHEMA BT_DQ_REF;

## Objects

`load_ts` is `TIMESTAMP_LTZ`. A direct `FRESHNESS` call accepts `DATE`, `TIMESTAMP_LTZ`, and `TIMESTAMP_TZ`, and rejects `TIMESTAMP_NTZ`.

In [ ]:
CREATE TABLE IF NOT EXISTS bt_batch_id (
  id NUMBER NOT NULL
)
COMMENT = 'Monotonic batch id. Latest attempt is MAX(bt_load_ledger.batch_id), never MAX(bt_orders.batch_id).';

In [ ]:
INSERT INTO bt_batch_id (id)
SELECT 0
WHERE NOT EXISTS (SELECT 1 FROM bt_batch_id);

In [ ]:
CREATE TABLE IF NOT EXISTS bt_orders (
  order_id NUMBER        NOT NULL,
  line_no  NUMBER        NOT NULL,
  sku      VARCHAR       NOT NULL,
  qty      NUMBER        NOT NULL,
  load_ts  TIMESTAMP_LTZ NOT NULL,
  batch_id NUMBER        NOT NULL
)
COMMENT = 'Synthetic orders. Business key is (order_id, line_no). load_ts is not part of the key.';

In [ ]:
CREATE TABLE IF NOT EXISTS bt_load_ledger (
  batch_id     NUMBER        NOT NULL,
  batch_name   VARCHAR       NOT NULL,
  attempted_at TIMESTAMP_NTZ NOT NULL,
  row_count    NUMBER        NOT NULL,
  note         VARCHAR
)
COMMENT = 'One row per load attempt, including empty loads.';

## Views

Each view filters to the ledger's newest batch id. They are separate objects because every DMF on one object shares that object's schedule, and the row-count check and the freshness check run on different cadences.

In [ ]:
CREATE OR REPLACE VIEW bt_orders_latest_batch AS
SELECT o.order_id, o.line_no, o.sku, o.qty, o.load_ts, o.batch_id
FROM bt_orders o
WHERE o.batch_id = (SELECT MAX(l.batch_id) FROM bt_load_ledger l);

In [ ]:
CREATE OR REPLACE VIEW bt_orders_latest_freshness AS
SELECT o.order_id, o.line_no, o.load_ts, o.batch_id
FROM bt_orders o
WHERE o.batch_id = (SELECT MAX(l.batch_id) FROM bt_load_ledger l);

In [ ]:
SHOW TABLES;
SHOW VIEWS;